# 02 – Feature Engineering (FIXED)

**Purpose:** Transform raw cleaned data into model‑ready features.

**Major changes:**
- Normalise fertilizer/pesticide by area → `Fertilizer_per_ha`, `Pesticide_per_ha`
- Log area → `log_Area`
- Target‑encode `Crop` (mean yield per crop, then log)
- One‑hot encode `Season` (remove ordinal lie)
- Remove extreme coconut outliers (Yield > 1000)
- Drop `Phosphorus`, `Potassium` (near‑constant after state aggregation)
- New feature set (13 features) – model will auto‑adapt input dimension

In [52]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import os

os.makedirs('../backend/models', exist_ok=True)
pd.set_option('display.max_columns', 20)

In [53]:
df = pd.read_csv('../data/cleaned_training_data.csv')
print("Shape:", df.shape)
df.head(3)

Shape: (19689, 14)


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield,Nitrogen,Phosphorus,Potassium,Soil Ph
0,Arecanut,1997,Whole Year,ASSAM,73814.0,56708,2051.4,7024878.38,22882.34,0.796087,50.542993,50.533311,50.482154,50.57468
1,Arhar/Tur,1997,Kharif,ASSAM,6637.0,4685,2051.4,631643.29,2057.47,0.710435,50.542993,50.533311,50.482154,50.57468
2,Castor seed,1997,Kharif,ASSAM,796.0,22,2051.4,75755.32,246.76,0.238333,50.542993,50.533311,50.482154,50.57468


In [54]:
print("Missing values:\n", df.isnull().sum())

Missing values:
 Crop               0
Crop_Year          0
Season             0
State              0
Area               0
Production         0
Annual_Rainfall    0
Fertilizer         0
Pesticide          0
Yield              0
Nitrogen           0
Phosphorus         0
Potassium          0
Soil Ph            0
dtype: int64


### 1. Feature engineering (area normalisation, target encoding, one‑hot)

In [55]:
# ---- Area normalisation ----
df['Fertilizer_per_ha'] = df['Fertilizer'] / df['Area']
df['Pesticide_per_ha']  = df['Pesticide']  / df['Area']
df['log_Area']          = np.log1p(df['Area'])

In [56]:
# ---- Target encoding for Crop (55 classes) ----
crop_mean_yield = df.groupby('Crop')['Yield'].mean()
df['Crop_target_enc'] = np.log1p(df['Crop'].map(crop_mean_yield))
joblib.dump(crop_mean_yield, '../backend/models/crop_target_map.pkl')
print("Crop target encoding saved.")

Crop target encoding saved.


In [57]:
# ---- Reverse mapping from encoded crop value (log1p(mean yield)) to crop name ----
# Round to avoid floating point drift during inference
crop_encoded_to_name = {round(np.log1p(v), 6): k for k, v in crop_mean_yield.items()}
joblib.dump(crop_encoded_to_name, '../backend/models/crop_encoded_to_name.pkl')
print("Crop encoded -> name mapping saved.")

Crop encoded -> name mapping saved.


In [58]:
# ---- One‑hot encode Season (6 classes → 5 dummies) ----
season_dummies = pd.get_dummies(df['Season'], prefix='Season', drop_first=True)
df = pd.concat([df, season_dummies], axis=1)
print("Season dummies added. Columns:", list(season_dummies.columns))

Season dummies added. Columns: ['Season_Kharif     ', 'Season_Rabi       ', 'Season_Summer     ', 'Season_Whole Year ', 'Season_Winter     ']


In [59]:
# ---- Remove extreme coconut outliers (Yield > 1000) ----
removed = len(df[df['Yield'] > 1000])
df = df[df['Yield'] < 1000].reset_index(drop=True)
print(f"Removed {removed} rows with Yield > 1000 (coconut noise).")

Removed 164 rows with Yield > 1000 (coconut noise).


In [60]:
# ---- Crop median yields (baseline point estimate) ----
crop_medians = df.groupby('Crop')['Yield'].median()
joblib.dump(crop_medians, '../backend/models/crop_medians.pkl')
print("Crop medians saved.")

Crop medians saved.


### 2. Final feature list (13 features)
- Crop_target_enc (target‑encoded)
- 5 Season dummies
- State (label encoded, still categorical)
- Annual_Rainfall
- Fertilizer_per_ha, Pesticide_per_ha, log_Area
- Nitrogen, Soil Ph
- (Phosphorus & Potassium dropped – state‑averaged, near‑constant)

In [61]:
season_cols = [c for c in df.columns if c.startswith('Season_')]
feature_cols = ['Crop_target_enc'] + season_cols + [
    'State', 'Annual_Rainfall',
    'Fertilizer_per_ha', 'Pesticide_per_ha', 'log_Area',
    'Nitrogen', 'Soil Ph'
]
target_col = 'Yield'

X = df[feature_cols].copy()
y = df[target_col].values

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Feature columns:", X.columns.tolist())

Features shape: (19525, 13)
Target shape: (19525,)
Feature columns: ['Crop_target_enc', 'Season_Kharif     ', 'Season_Rabi       ', 'Season_Summer     ', 'Season_Whole Year ', 'Season_Winter     ', 'State', 'Annual_Rainfall', 'Fertilizer_per_ha', 'Pesticide_per_ha', 'log_Area', 'Nitrogen', 'Soil Ph']


### 3. Encode categorical `State` (only categorical left)

In [62]:
le_state = LabelEncoder()
X['State'] = le_state.fit_transform(X['State'].astype(str))
joblib.dump(le_state, '../backend/models/le_state.pkl')
print("State encoded, saved.")

State encoded, saved.


### 4. Scale numerical features (except dummy dummies)

In [63]:
numerical_cols = ['Annual_Rainfall', 'Fertilizer_per_ha', 'Pesticide_per_ha',
                  'log_Area', 'Nitrogen', 'Soil Ph', 'Crop_target_enc']

scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])
joblib.dump(scaler, '../backend/models/scaler.pkl')
print("Scaler saved.")
print("Numerical means after scaling:\n", X[numerical_cols].mean().round(2))

Scaler saved.
Numerical means after scaling:
 Annual_Rainfall     -0.0
Fertilizer_per_ha   -0.0
Pesticide_per_ha     0.0
log_Area             0.0
Nitrogen            -0.0
Soil Ph             -0.0
Crop_target_enc     -0.0
dtype: float64


### 5. Train / validation / test split

In [64]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=X['State']
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=X_temp['State']
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (13667, 13), Val: (2929, 13), Test: (2929, 13)


In [65]:
# ---- Save crop names for test set (used later for baseline) ----
# Get the crop names from the original dataframe for the test indices
# We need to map back from the split indices to the original dataframe.

# First, get the original indices from the split
# X and y are aligned with df, we used train_test_split with stratify on 'State'
# So we can recover test indices by tracking original index.

# Better: during split, keep the original index.
# But our split used X and y which are NumPy arrays without index.
# Instead, we can re‑extract test crop names by using the boolean mask.

# Re‑create the split using DataFrames to preserve crop column:
from sklearn.model_selection import train_test_split

# Use the same random_state and stratify as before
X_df = df[feature_cols + ['Crop']].copy()   # include Crop temporarily
y_df = df['Yield']

X_train_df, X_temp_df, y_train, y_temp = train_test_split(
    X_df, y, test_size=0.3, random_state=42, stratify=X_df['State']
)
X_val_df, X_test_df, y_val, y_test = train_test_split(
    X_temp_df, y_temp, test_size=0.5, random_state=42, stratify=X_temp_df['State']
)

# Now extract test crop names
test_crop_names = X_test_df['Crop'].values

# Save them
np.save('../data/test_crop_names.npy', test_crop_names)
print("Test crop names saved.")

Test crop names saved.


### 6. Save processed arrays

In [66]:
# Convert to float32 before saving (critical for PyTorch)
X_train_arr = X_train.values.astype(np.float32)
X_val_arr = X_val.values.astype(np.float32)
X_test_arr = X_test.values.astype(np.float32)
y_train_arr = y_train.astype(np.float32)
y_val_arr = y_val.astype(np.float32)
y_test_arr = y_test.astype(np.float32)

np.save('../data/X_train.npy', X_train_arr)
np.save('../data/y_train.npy', y_train_arr)
np.save('../data/X_val.npy', X_val_arr)
np.save('../data/y_val.npy', y_val_arr)
np.save('../data/X_test.npy', X_test_arr)
np.save('../data/y_test.npy', y_test_arr)

print("All arrays saved as float32.")

print("All arrays saved.")

All arrays saved as float32.
All arrays saved.
